# GeoFWI3D — bounded UNETR rescue experiment

Two recipes, seed 0 / fold 0. Both use **dropout 0.1 + foreground-only squared Dice + CE (1:1)**.
Arm A peak LR **1e-4**, arm B **3e-5**. Same 92,802,274 parameters, fresh identical initialization,
19 training / 96 validation cases, native 96³, BF16, batch 1 and **no augmentation**.

Each arm first runs **207 updates**, at most half the original cap. Select by mean validation Dice
at 30%, 40%, and 50% of budget (tie -> A). Resume that arm to **414 total updates** with its
optimizer/RNG and full learning-rate schedule intact. The other arm is recorded as pruned.

The CPU full-model profile measured **439.528930661 GFLOPs/update**, including the new loss and dropout.
Maximum planned optimization: **272,947.466 GFLOPs**, below the authorized
**273,267.190 GFLOP** total. GPU preflight uses six diagnostic updates in total;
validation/setup costs are outside the optimization cap and reported separately.

Success requires **final-budget validation Dice >50%**, not simply winning this race.
This is validation-guided recipe development, not completion of the three-seed/three-fold baseline.
Half-budget screening can miss a slower learner. No automatic extra seeds or follow-on experiments.

Account: **focacciafilippo@gmail.com**. Use **A100-SXM4-80GB + High RAM**, matching the control.
Run only one copy. Old pilot results remain in their separate results folder.


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from googleapiclient.discovery import build
import google.auth
credentials, _ = google.auth.default()
service = build('drive','v3',credentials=credentials,cache_discovery=False)
about = service.about().get(fields='user(emailAddress),storageQuota').execute()
assert about['user']['emailAddress'].lower() == 'focacciafilippo@gmail.com', 'Mount/authenticate the requested account.'
quota=about['storageQuota']
if 'limit' in quota:
    assert int(quota['limit'])-int(quota['usage']) >= 8*1024**3, 'Keep at least 8 GiB free in Drive, including trash.'
from pathlib import Path
import hashlib,json,os,shutil,signal,subprocess,sys,time,zipfile
HOME_DIR=Path('/content/drive/MyDrive/hippo_geofwi3d_unetr_ce_pilot_20261019')
INPUT=HOME_DIR/'input'
RESULTS=HOME_DIR/'results_unetr_rescue_f0s0_v1'
BUNDLE=Path('/content/hippo_geofwi_unetr_rescue_v1')
RESULTS.mkdir(parents=True,exist_ok=True)
BUNDLE.mkdir(exist_ok=True)
assert shutil.disk_usage('/content').free >= 24*1024**3, 'Need at least 24 GiB runtime disk space.'
print('Account:',about['user']['emailAddress'],'Results:',RESULTS)


## Verify and stage the uploaded data/code
All archive and per-file SHA256 hashes are checked. The 319 unique images/labels are read from
the runtime SSD during training. Existing matching files are reused; changed files stop execution.


In [ ]:
EXPECTED_UPLOADS={'geofwi3d_unetr_rescue_code_v1.zip': 'ff251c83284bff641445ca386c1b93aa1fea21071464084d9ead9eaeb4c27809', 'geofwi3d_data_part01.zip': '32eaadf1a6abd61dd5bd922a8a085af4e6c876b34b915bbd09da93a3dbea2ca2', 'geofwi3d_data_part02.zip': '31adc6fa680b74a921f9457f24d2dfbb178aa1cef1d80baa152c5ee82a4e05bd', 'geofwi3d_data_part03.zip': 'e9c404631c47a918d1ca36c9166ea92141aa1b8264e332fbaa32b13299d8706a', 'geofwi3d_data_part04.zip': '1ef1e965ebefb571f441d986f2304edb4f2ab824b4bedb7dfbc6a494c4c1c279', 'geofwi3d_data_part05.zip': 'b99cbe88932b94bc382512c059661b48516beacd1dae116237ec411edc64fb58', 'geofwi3d_data_part06.zip': '6f0509326647688300459b05d5f0af5c10ecdde9c19fc6fc3ee05bc92fe8f183', 'geofwi3d_data_part07.zip': '3e1cf8a1cacd445c73bf1798ebd9c1b3c34f6f2cea0e2bf0ad378c6f12f09fb1', 'geofwi3d_data_part08.zip': 'b2b603f5ef55eb2822e64904ce7d178b7cd459c1258398de1f75cdedad5525ed', 'geofwi3d_data_part09.zip': '833fccd13996d6d6ddd118b6146334a902e4eceba2dedd26b798db64fcf6de38', 'geofwi3d_data_part10.zip': '99daf1d8f684ac47897aaa6933b69741e826935783bc42dbd02093efc299e235', 'geofwi3d_data_part11.zip': 'ce12340263414416fcd728e7fad43a2dbbbf4d26f22cc51273acb8df50680239', 'geofwi3d_data_part12.zip': 'ff0f922b1ae82afb2394725bcd0374b4eb273fe0333f5fbad4dbd52b540aadd2', 'geofwi3d_data_part13.zip': '9ede65d6da572c0186b51b838f46f7cd281bbd8b515aef813e502b59891cc880'}
EXPECTED_MANIFEST='ac567d173e52b8a96cd0a44a23917df4f8c7abe320667e65ff21d4ae1a1057a2'

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024**2),b''):h.update(chunk)
    return h.hexdigest()
for name,expected in EXPECTED_UPLOADS.items():
    package=INPUT/name
    if not package.exists():raise FileNotFoundError(str(package))
    assert sha(package)==expected, 'Upload checksum mismatch: '+name
    local=Path('/content')/name
    if not local.exists() or sha(local)!=expected:
        shutil.copyfile(package,local)
    assert sha(local)==expected
    with zipfile.ZipFile(local) as z:
        assert len(z.namelist())==len(set(z.namelist())), 'Duplicate archive members'
        for info in z.infolist():
            p=Path(info.filename)
            assert not p.is_absolute() and '..' not in p.parts
            assert (info.external_attr>>16)&0o170000 != 0o120000
        for info in z.infolist():
            target=BUNDLE/info.filename
            if target.exists():continue
            target.parent.mkdir(parents=True,exist_ok=True)
            temp=target.with_name(target.name+'.extracting')
            with z.open(info) as src,temp.open('wb') as dst:shutil.copyfileobj(src,dst)
            temp.replace(target)
    print('Staged',name)
assert sha(BUNDLE/'bundle_manifest.json')==EXPECTED_MANIFEST
for listing in ['bundle_manifest.json','data_manifest.json']:
    for name,expected in json.loads((BUNDLE/listing).read_text()).items():
        assert sha(BUNDLE/name)==expected, 'Extracted file mismatch: '+name
source=RESULTS/'provenance'/EXPECTED_MANIFEST;source.mkdir(parents=True,exist_ok=True)
for name in ['geofwi3d_unetr_rescue_code_v1.zip']:
    dst=source/name
    if not dst.exists():shutil.copyfile(INPUT/name,dst)
    assert sha(dst)==EXPECTED_UPLOADS[name]
print('All inputs verified. No augmentation.')


## Install the frozen environment
Uses the package versions from the existing Colab training environment, in a separate venv.
Setup output is streamed and saved in Drive. Keep the same GPU type/software on resume.


In [ ]:
VENV=Path('/content/hippo_geofwi_unetr_ce_pilot_venv');PYTHON=VENV/'bin/python'
def command(args,log_name):
    with (RESULTS/log_name).open('a') as log:
        proc=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,bufsize=1,start_new_session=True)
        tail=[]
        def drain():
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
                tail.append(line);del tail[:-40]
        try:drain();code=proc.wait()
        except KeyboardInterrupt:
            # Signal the orchestrator only; it stops the active trainer at a safe boundary.
            proc.send_signal(signal.SIGINT)
            print('Saving recovery to Drive. Please wait; do not interrupt again.')
            drain();code=proc.wait()
        finally:proc.stdout.close()
        if code:raise RuntimeError('Process failed; recovery retained.\n'+''.join(tail))
if not PYTHON.exists():
    command([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV],'setup.log')
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    command([PYTHON,'-m','ensurepip','--upgrade'],'setup.log')
command([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'],'setup.log')
def campaign(action,*extra):
    command([PYTHON,'-u',BUNDLE/'rescue_runtime.py',action,'--bundle',BUNDLE,'--durable',RESULTS,*extra],action+'.log')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)


## Required GPU checks
For each learning rate: native 96³ forward/backward, finite gradients and exact model/optimizer/RNG
checkpoint replay with dropout active. Six diagnostic optimizer updates in total, recorded separately.
Initialization must match the archived seed-0 control. No research updates occur before these checks pass.


In [ ]:
campaign('preflight')


## Run or resume the bounded experiment
Runs A to half budget, B to half budget, records the selection, then resumes only the winner to full budget.
The timer pauses safely; rerunning reuses completed stages. Interrupt once and wait for recovery upload.
A hard VM loss with uncheckpointed charged work stops for an audit rather than silently replaying it.
All scientific completion and pruning records are separate from session pauses and technical failures.


In [ ]:
SESSION_HOURS=2.0
# Create RESULTS/'STOP' to request a graceful stop.
campaign('run','--hours',str(SESSION_HOURS))


## Results
`rescue_status.json` contains both screening scores, the selected arm, charged compute and the winner's
final result. `RESCUE_COMPLETE.json` means this development experiment finished; `threshold_passed`
states whether final validation Dice is strictly above 50%. It never declares the 3×3 baseline complete.
Curves, confusion counts, source, config and logs are verified in Drive before obsolete checkpoints retire.


In [ ]:
campaign('report')
import pandas as pd
status=json.loads((RESULTS/'rescue_status.json').read_text())
print('Status:',status['status'])
display(pd.DataFrame(status['screening']))
print('Selection:',status['selection'])
print('Training GFLOPs:',status['total_training_flops']/1e9)
if status['final']:
    r=status['final']
    print('Final validation Dice:',100*r['final_dice'],'%')
    print('Best validation Dice:',100*r['best_dice'],'%')
    print('Final training Dice:',100*r['training_dice'],'%')
    print('Final validation Dice >50%:',status['threshold_passed'])
else:print('Not complete; inspect status/logs before resuming.')
